# glcuda probe

Everything the T4 can answer that the dev machine cannot, in one pass. Terse by
design; the reasoning lives in git history, not in this notebook.

**Guards that are not optional** — each one caught a wrong conclusion once:

* `--test-threads=1` — `capture` uses `CU_STREAM_CAPTURE_MODE_GLOBAL`, so
  parallel tests fail with `STREAM_CAPTURE_UNSUPPORTED` and look like
  regressions.
* parity **SKIPs** without a GPU, so a green summary can mean nothing ran.
* the A/B checks the engine's own banner — two identical arms report a clean 0%.
* the A/B discards one warmup sweep; the first run of a process was the highest
  of its arm in *both* arms.


## 1 · setup

In [ ]:
REPO_URL, BRANCH, GH_TOKEN = "https://github.com/gwenland-org/gwenland-ai.git", "glbench-vs-llamacpp", ""
MODEL_REPO = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main"
MODEL_FILE = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
GEN_TOKENS, WARMUP, ITERS = 128, 3, 10
AB_ENV, AB_BANNER, AB_MARKER = ({"GLCUDA_R256": "1"}, "r256 prefill GEMM enabled",
                                ("glcuda/src/kernels/mod.rs", "GLCUDA_R256"))
AB_REPEATS, AB_WARMUP = 4, 1

import os, re, json, time, shutil, statistics, subprocess
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO_DIR, OUT_DIR = os.path.join(WORK, "gwenland-ai"), WORK
os.makedirs(WORK, exist_ok=True)

def sh(cmd, cwd=None, timeout=7200, env=None):
    e = dict(os.environ); e.update(env or {})
    try:
        p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True,
                           timeout=timeout, env=e, stdin=subprocess.DEVNULL)
        return p.returncode, p.stdout, p.stderr
    except subprocess.TimeoutExpired: return 124, "", "timeout"
    except Exception as ex: return 125, "", f"{type(ex).__name__}: {ex}"

rc, out, _ = sh(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], timeout=60)
GPUS = [l.strip() for l in out.splitlines() if l.strip()] if rc == 0 else []
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(f"gpu     {len(GPUS)}x {GPUS[0] if GPUS else 'NONE'} -> pinned dev 0")
if not GPUS: print("        ⛔ no GPU: parity SKIPs and every gate below is vacuous")

# repo
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    rc1, _, e1 = sh(["git", "fetch", "--depth", "50", "origin", BRANCH], cwd=REPO_DIR)
    rc2, _, e2 = sh(["git", "reset", "--hard", "FETCH_HEAD"], cwd=REPO_DIR)
    if rc1 or rc2: print("        ⛔ refresh FAILED:", (e1 or e2)[:150])
else:
    url = REPO_URL.replace("https://", f"https://{GH_TOKEN}@") if GH_TOKEN else REPO_URL
    sh(["git", "clone", "--depth", "50", "--branch", BRANCH, url, REPO_DIR])
_, o, _ = sh(["git", "log", "--oneline", "-1"], cwd=REPO_DIR)
COMMIT = o.strip()
_, o2, _ = sh(["git", "rev-parse", "--short", f"origin/{BRANCH}"], cwd=REPO_DIR)
print(f"commit  {COMMIT[:60]}")
if o2.strip() and not COMMIT.startswith(o2.strip()[:7]):
    print(f"        ⛔ BEHIND {BRANCH} (tip {o2.strip()}) -- measuring older code")

if shutil.which("cargo") is None:
    os.system("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal >/dev/null 2>&1")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

# model
MODEL = next((p for p in [os.path.join(WORK, MODEL_FILE)]
              if os.path.exists(p) and os.path.getsize(p) > 1e7), None)
if not MODEL:
    for root, _, fs in os.walk("/kaggle/input") if os.path.isdir("/kaggle/input") else []:
        for f in fs:
            if f.lower().endswith(".gguf") and "0.5b" in f.lower():
                MODEL = os.path.join(root, f); break
        if MODEL: break
if not MODEL:
    MODEL = os.path.join(WORK, MODEL_FILE)
    sh(["curl", "-fL", "--retry", "3", "-o", MODEL, f"{MODEL_REPO}/{MODEL_FILE}"], timeout=1800)
assert os.path.exists(MODEL) and os.path.getsize(MODEL) > 1e7, "no model"
print(f"model   {os.path.basename(MODEL)}  {os.path.getsize(MODEL)/1e9:.3f} GB")


## 2 · build + reference run

In [ ]:
GL_BIN = os.path.join(REPO_DIR, "target/release/glbench")
CACHE = os.path.join(WORK, "glbench-bin-cache")
man_p = os.path.join(CACHE, "manifest.json")

restored = False
if os.path.exists(man_p):
    try:
        if json.load(open(man_p)).get("commit") == COMMIT and os.path.exists(os.path.join(CACHE, "glbench")):
            os.makedirs(os.path.dirname(GL_BIN), exist_ok=True)
            shutil.copy2(os.path.join(CACHE, "glbench"), GL_BIN); os.chmod(GL_BIN, 0o755)
            restored = True
    except Exception: pass
if not restored:
    t = time.time()
    rc, o, e = sh(["cargo", "build", "--release", "-p", "glbench"], cwd=REPO_DIR)
    assert rc == 0, (o + e)[-3000:]
    os.makedirs(CACHE, exist_ok=True)
    shutil.copy2(GL_BIN, os.path.join(CACHE, "glbench"))
    json.dump({"commit": COMMIT}, open(man_p, "w"))
    print(f"build   {(time.time()-t)/60:.1f} min")
else:
    print("build   cached")

GL_CMD = [GL_BIN, "run", "--engine", "glcuda", "--model", MODEL, "--tokens", str(GEN_TOKENS),
          "--warmup", str(WARMUP), "--iters", str(ITERS), "--out", os.path.join(OUT_DIR, "ref.json")]
rc, REF_OUT, REF_ERR = sh(GL_CMD, cwd=REPO_DIR)
assert rc == 0, REF_ERR[-2000:]
REF = json.load(open(os.path.join(OUT_DIR, "ref.json")))
AN = REF.get("analysis") or {}
its = (REF.get("measurements") or {}).get("iterations") or []
PROMPT_TOKENS = int(its[0]["prompt_tokens"]) if its else None
PRE, DEC = (AN.get("prefill_tps") or {}), (AN.get("decode_tps") or {})
print(f"\n== reference (unprofiled) ==")
print(f"prompt  {PROMPT_TOKENS} tok")
print(f"prefill {PRE.get('mean',0):.1f} tok/s  (median {PRE.get('median',0):.1f}, sd {PRE.get('std_dev',0):.1f})")
print(f"decode  {DEC.get('mean',0):.1f} tok/s  (median {DEC.get('median',0):.1f}, sd {DEC.get('std_dev',0):.1f})")
print(f"verdict {AN.get('bottleneck')}  {('%.0f%%' % (AN['ceiling_efficiency']*100)) if AN.get('ceiling_efficiency') else '-'} of ceiling")


## 3 · parity on hardware

In [ ]:
rc, o, e = sh(["cargo", "test", "-p", "glcuda", "--release", "--", "--test-threads=1"],
              cwd=REPO_DIR)
T = o + "\n" + e
open(os.path.join(OUT_DIR, "tests.txt"), "w").write(T)
SKIPPED = "SKIP: no CUDA driver/device" in T

def verdict_of(name):
    line = None
    for l in T.splitlines():
        s = l.strip()
        if s.startswith("test ") and name in s and " ... " in s: line = s
    if SKIPPED: return "SKIPPED (no device -- proves nothing)"
    if line and line.endswith(" ok"): return "PASS"
    if line and "FAILED" in line: return "FAIL"
    return f"? {line or 'not found'}"

print("== parity ==")
for n in ["gemm_mma_q8_matches", "gemm_mma_q8_r256_matches"]:
    print(f"{n:28s} {verdict_of(n)}")
d = re.search(r"^gemm_mma_q8_r256\([^\n]*", T, re.M)
if d: print("  ", d.group(0).strip())
print("summary ", " | ".join(l for l in T.splitlines() if l.startswith("test result:")))
PARITY_OK = verdict_of("gemm_mma_q8_r256_matches") == "PASS"


## 4 · profiles (medians, not every iteration)

In [ ]:
BUCKETS = ["qkv", "attn", "ffn", "attn core", "gate+up GEMM", "down+o GEMM", "elementwise"]

def split_ms(hay):
    out = {}
    for b in BUCKETS:
        v = [int(m) for m in re.findall(re.escape(b) + r"\s+(\d+)ms", hay)]
        if v: out[b] = statistics.median(v)
    return out

rc, o, e = sh(GL_CMD[:-1] + [os.path.join(OUT_DIR, "prof.json")], cwd=REPO_DIR,
              env={"GLCUDA_PROFILE_PREFILL": "1"})
P_HAY = o + "\n" + e
SPLIT = split_ms(P_HAY)
print("== prefill buckets (median ms over iterations) ==")
if SPLIT:
    tot = SPLIT.get("qkv", 0) + SPLIT.get("attn", 0) + SPLIT.get("ffn", 0)
    for b in BUCKETS:
        if b in SPLIT:
            print(f"  {b:14s} {SPLIT[b]:6.1f} ms  {100*SPLIT[b]/tot if tot else 0:5.1f}%")
else:
    print("  UNMEASURED")

rc, o, e = sh(GL_CMD[:-1] + [os.path.join(OUT_DIR, "dec.json")], cwd=REPO_DIR,
              env={"GLCUDA_PROFILE_DECODE": "1"})
m = re.search(r"\[decode split\][^\n]*", o + "\n" + e)
print("\n== decode split ==")
if m:
    print(" ", m.group(0))
    g = re.search(r"GPU\s*([\d.]+) ms/tok", m.group(0))
    if g: print(f"  GPU-only {1000/float(g.group(1)):.1f} tok/s  vs full pipeline {DEC.get('mean',0):.1f} tok/s")
else:
    print("  UNMEASURED")


## 5 · kernel probes

In [ ]:
rc, o, e = sh(["cargo", "build", "--release", "-p", "glcuda", "--example", "bench"], cwd=REPO_DIR)
assert rc == 0, (o + e)[-2000:]
rc, o, e = sh([os.path.join(REPO_DIR, "target/release/examples/bench")], cwd=REPO_DIR)
B = o + "\n" + e
open(os.path.join(OUT_DIR, "bench.txt"), "w").write(B)
pick = lambda tag: [l.strip() for l in B.splitlines() if tag in l and "=>" not in l[:20]]

print("== gemv vs gemm ==")
GEMV = {}
for l in pick("[gemv-vs-gemm"):
    print(" ", l[:150])
    n, r = re.search(r"\[gemv-vs-gemm (\w+)", l), re.search(r"GEMV/GEMM ([\d.]+)x", l)
    if n and r: GEMV[n.group(1)] = float(r.group(1))
for k_, v in GEMV.items():
    print(f"  {k_:6s} {v:.2f}x  " + ("blocks beat traffic -> split-K is the fix"
                                     if v < 1.0 else "traffic wins -> GEMM shape is not the problem"))

print("\n== ffn-context ladder ==")
FFN = [l.strip() for l in B.splitlines() if "[ffn-context" in l]
for l in FFN: print(" ", l[:150])

def rung(label):
    for l in FFN:
        if f"[ffn-context {label}" in l:
            return {int(m.group(1)): float(m.group(2)) for m in re.finditer(r"s(\d) ([\d.]+)us", l)}
    return {}
G, D = rung("gate"), rung("down")
FFN_VERDICT = "UNMEASURED"
if G and D and 0 in G and 0 in D:
    rows = []
    print("\n  step   gate              down              diff")
    for s in sorted(set(G) & set(D)):
        gp, dp = 100*(G[s]-G[0])/G[0], 100*(D[s]-D[0])/D[0]
        rows.append((s, dp - gp))
        print(f"  s{s}   {G[s]:8.1f}us {gp:+6.0f}%   {D[s]:8.1f}us {dp:+6.0f}%   {dp-gp:+7.0f} pts")
    jumps = [(rows[i][1] - rows[i-1][1], rows[i][0]) for i in range(1, len(rows))]
    bj, bs = max(jumps) if jumps else (0.0, None)
    reached = 100 * D[max(D)] / 822.9
    if bs is not None and bj >= 50:
        FFN_VERDICT = f"s{bs} separates them ({bj:+.0f} pts); ladder reaches {reached:.0f}% of prefill's 822.9us"
    elif reached < 50:
        FFN_VERDICT = f"NOT REPRODUCED ({reached:.0f}% of prefill, max sep {bj:.0f} pts) -- cause is outside these four"
    else:
        FFN_VERDICT = f"NO SINGLE STEP ({reached:.0f}% of prefill, max sep {bj:.0f} pts) -- cost accumulates"
print("\n ", FFN_VERDICT)

for tag in ["[gemm-phaseb", "[r256-parity", "[r256-ladder"]:
    ls = pick(tag)
    if ls:
        print(f"\n== {tag[1:]} ==")
        for l in ls: print(" ", l[:150])


## 6 · A/B (interleaved, warmup discarded, overlap rule)

In [ ]:
_src = os.path.join(REPO_DIR, *AB_MARKER[0].split("/"))
HAVE = os.path.exists(_src) and AB_MARKER[1] in open(_src, errors="replace").read()
if not HAVE:
    print(f"⛔ SKIPPED: {AB_MARKER[1]} absent from this checkout -- both arms would be identical")

ARMS = {"base": [], "var": []}
DISCARD, BANNER = [], None
for r in range(AB_REPEATS + AB_WARMUP if HAVE else 0):
    for arm, env in (("base", {}), ("var", AB_ENV)):
        out = os.path.join(OUT_DIR, f"ab_{arm}_{r}.json")
        rc, o, e = sh(GL_CMD[:-1] + [out], cwd=REPO_DIR, env=env)
        hay = o + "\n" + e
        if BANNER is None and AB_BANNER in hay:
            BANNER = re.search(r"\[glcuda\][^\n]*" + re.escape(AB_BANNER) + r"[^\n]*", hay).group(0)
        try: tps = (json.load(open(out))["analysis"]["prefill_tps"]["mean"])
        except Exception: tps = None
        (DISCARD if r < AB_WARMUP else ARMS[arm]).append(tps)

print("== A/B: r256 ==")
if DISCARD: print("  warmup discarded:", " ".join(f"{v:.0f}" for v in DISCARD if v))
print("  banner:", BANNER or "NOT SEEN -- flag never took effect")
def stat(a):
    v = sorted(x for x in ARMS[a] if x)
    return (v[0], statistics.median(v), v[-1]) if v else (None, None, None)
bl, bm, bh = stat("base"); vl, vm, vh = stat("var")
if bm and vm:
    print(f"  base  {bl:.0f} - {bh:.0f}   median {bm:.0f}")
    print(f"  r256  {vl:.0f} - {vh:.0f}   median {vm:.0f}   ({100*(vm-bm)/bm:+.1f}%)")

if not HAVE:                 AB_VERDICT = "NOT RUN (code absent)"
elif BANNER is None:         AB_VERDICT = "INVALID (banner never appeared -- arms may be identical)"
elif not (bm and vm):        AB_VERDICT = "UNMEASURED"
elif vl > bh:                AB_VERDICT = f"KEEP (worst r256 {vl:.0f} > best base {bh:.0f}; no overlap)"
elif vh < bl:                AB_VERDICT = f"REGRESSION (best r256 {vh:.0f} < worst base {bl:.0f})"
else:                        AB_VERDICT = f"INCONCLUSIVE (overlap; median {100*(vm-bm)/bm:+.1f}% = this machine's drift)"
print("\n ", AB_VERDICT)


## 7 · summary

In [ ]:
L = []
L.append(f"# glcuda probe · {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}")
L.append(f"`{COMMIT}`  ·  {len(GPUS)}x {GPUS[0] if GPUS else '-'}  ·  {PROMPT_TOKENS} tok prompt")
L.append("")
L.append("| | |")
L.append("|---|---|")
L.append(f"| prefill | {PRE.get('mean',0):.1f} tok/s |")
L.append(f"| decode | {DEC.get('mean',0):.1f} tok/s |")
L.append(f"| r256 parity | {verdict_of('gemm_mma_q8_r256_matches')} |")
L.append(f"| A/B r256 | {AB_VERDICT} |")
L.append(f"| ffn-context | {FFN_VERDICT} |")
for k_, v in GEMV.items():
    L.append(f"| gemv/gemm {k_} | {v:.2f}x |")
L.append("")
if SPLIT:
    L.append("## prefill buckets (median ms)")
    L.append("")
    L.append("| bucket | ms |")
    L.append("|---|---:|")
    for b in BUCKETS:
        if b in SPLIT: L.append(f"| {b} | {SPLIT[b]:.0f} |")
    L.append("")
L.append("## raw")
L.append("")
for t, body in [("bench", B), ("prefill profile", P_HAY), ("tests", T)]:
    L.append(f"<details><summary>{t}</summary>")
    L.append("")
    L.append("```")
    L.append((body or "").strip()[:12000])
    L.append("```")
    L.append("")
    L.append("</details>")
    L.append("")
p = os.path.join(OUT_DIR, "GLCUDA_PROBE.md")
open(p, "w", encoding="utf-8").write("\n".join(L))

print("=" * 58)
print(f"prefill      {PRE.get('mean',0):.1f} tok/s      decode {DEC.get('mean',0):.1f} tok/s")
print(f"r256 parity  {verdict_of('gemm_mma_q8_r256_matches')}")
print(f"r256 A/B     {AB_VERDICT}")
print(f"ffn-context  {FFN_VERDICT}")
for k_, v in GEMV.items(): print(f"gemv/gemm    {k_} {v:.2f}x")
print("=" * 58)
print(p)
